In [ ]:
from pulao.bar import SBar
import logging

from pulao.keyzone import KeyZoneManager
from pulao.logging import init_logging
from pulao.mtc import MultiTimeframeContext
from datetime import datetime, time, date

from pulao.constant import Timeframe,Exchange

import polars as pl

from pulao.logging import get_logger
from pulao.symbol import SymbolLoader

init_logging(level=logging.DEBUG)
open("./logs/pulao.log", "w").close()  # 每次运行前清空日志
logger = get_logger(__name__)

logger.debug("开始运行")
SymbolLoader.load("../pulao/symbol/data/symbols.yaml")
symbol = "i8888"
dataset = [
    dict(timeframe=Timeframe.M15, filename="../dataset/I8888.XDCE_15m.csv", length=30,data=[]),
    dict(timeframe=Timeframe.H1, filename="../dataset/I8888.XDCE_60m.csv", length=30, data=[]),
    dict(timeframe=Timeframe.D1, filename="../dataset/I8888.XDCE_1d.csv", length=30, data=[]),
]
for d in dataset:
    timeframe = d.get("timeframe")
    filename = d.get("filename")
    length = d.get("length")
    df = pl.read_csv(filename, try_parse_dates=True)
    # 重新采样
    df = df.group_by_dynamic(
        index_column="datetime",
        every=timeframe.value,
        closed="left"
    ).agg([
        pl.first("open").alias("open"),
        pl.max("high").alias("high"),
        pl.min("low").alias("low"),
        pl.last("close").alias("close"),
        pl.sum("volume").alias("volume"),
        pl.sum("money").alias("money"),
        pl.last("open_interest").alias("open_interest")
    ]).drop_nulls()
    df = df.slice(0,length)  #TEST 取多少条数据用于测试

    sbar_list = d.get("data")
    columns = df.columns

    for idx, row in enumerate(df.iter_rows()):
        row_dict = dict(zip(columns, row))
        # datetime,open,close,high,low,volume,money,open_interest,signal
        dt = row_dict["datetime"]
        if not isinstance(dt, datetime):
            dt = datetime.combine(dt, time())

        o = row_dict["open"]
        close = row_dict["close"]
        high = row_dict["high"]
        low = row_dict["low"]
        volume = row_dict["volume"]
        money = row_dict["money"]
        open_interest = row_dict["open_interest"]

        sbar = SBar(
            symbol=symbol,
            exchange=Exchange.SHFE,
            timeframe=timeframe,
            datetime=dt,
            turnover=money,
            open_price=o,
            close_price=close,
            high_price=high,
            low_price=low,
            volume=volume,
            open_interest=open_interest,
        )

        sbar_list.append(sbar)
# 模拟行情数据接收

# 周期：manager list

mtc = MultiTimeframeContext(symbol=symbol)
keyzone_manager = KeyZoneManager(mtc=mtc)

# 注入模拟数据
for ds in dataset:
    timeframe = ds.get("timeframe")
    mtc.register(timeframe)
    for sbar in ds.get("data"):
        mtc.append(timeframe, sbar)

logger.debug("运行结束")


: 

In [ ]:
from typing import Any
from pulao.events import Observable
from pulao.constant import *

import polars as pl
import numpy as np

mtc.get_manager(Timeframe.H1).swing_manager.df_swing


: 

In [ ]:
from pulao.symbol import SymbolLoader
from datetime import timedelta,datetime,time
from pulao.keyzone import KeyZone
# 画图
import polars as pl
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pulao.constant import *
from pulao.mtc import MultiTimeframeContext
from pulao.keyzone import KeyZoneManager
import chinese_calendar as cc

SymbolLoader.load("../pulao/symbol/data/symbols.yaml")
symbol = "i8888"
timeframe = Timeframe.M15
mtc = MultiTimeframeContext(symbol=symbol)
mtc.register(timeframe)
tf_mgr = mtc.get_manager(timeframe)
sbar_manager = tf_mgr.sbar_manager
cbar_manager = tf_mgr.cbar_manager
swing_manager = tf_mgr.swing_manager
trend_manager = tf_mgr.trend_manager
sbar_manager.read_parquet()
cbar_manager.read_parquet()
swing_manager.read_parquet()
trend_manager.read_parquet()
#
# region . Plotly
#

df_sbar = sbar_manager.df_sbar.to_pandas().set_index("datetime", drop=False)
if "index" in df_sbar.columns:
    del df_sbar["index"]
df_sbar.insert(0, "index", range(len(df_sbar)))

# region 构建波段数据
df_swing = swing_manager.df_swing.to_pandas()
df_swing["start_datetime"] = pd.Series(dtype="datetime64[ns]")
df_swing["end_datetime"] = pd.Series(dtype="datetime64[ns]")
if "index" in df_swing.columns:
    del df_swing["index"]
df_swing.insert(0, "index", range(len(df_swing)))

for i, row in enumerate(df_swing.itertuples()):
    df = df_sbar[
        (df_sbar["id"] >= row.sbar_start_id) & (df_sbar["id"] <= row.sbar_end_id)
    ]
    start_datetime = df.iloc[0]["datetime"]
    end_datetime = df.iloc[-1]["datetime"]
    df_swing.at[i, "start_datetime"] = start_datetime
    df_swing.at[i, "end_datetime"] = end_datetime

xs_swing = []
ys_swing = []
texts_swing = []
text_positions_swing = []
for i, row in enumerate(df_swing.itertuples()):
    xs_swing += [row.start_datetime, row.end_datetime, None]
    print("df_swing - "+str(i),row)
    start_index = df_sbar[(df_sbar["id"] == row.sbar_start_id)]["index"].iloc[0]
    end_index = df_sbar[(df_sbar["id"] == row.sbar_end_id)]["index"].iloc[0]
    if row.direction == Direction.UP:
        start_price = df_sbar[(df_sbar["id"] == row.sbar_start_id)]["low_price"].iloc[0]
        end_price = df_sbar[(df_sbar["id"] == row.sbar_end_id)]["high_price"].iloc[0]
        text_positions_swing += ["bottom center", "top center", "middle center"]
    else:
        start_price = df_sbar[(df_sbar["id"] == row.sbar_start_id)]["high_price"].iloc[
            0
        ]
        end_price = df_sbar[(df_sbar["id"] == row.sbar_end_id)]["low_price"].iloc[0]
        text_positions_swing += ["top center", "bottom center", "middle center"]
    ys_swing += [start_price, end_price, None]
    texts_swing += [start_index, end_index, None]
# endregion

# region 构建趋势数据
df_trend = trend_manager.df_trend.to_pandas()
df_trend["start_datetime"] = pd.Series(dtype="datetime64[ns]")
df_trend["end_datetime"] = pd.Series(dtype="datetime64[ns]")
df_trend.insert(0, "index", range(len(df_trend)))

for i, row in enumerate(df_trend.itertuples()):
    print("df_trend[datatime] - "+ str(i), row)
    df = df_swing[
        (df_swing["id"] >= row.swing_start_id) & (df_swing["id"] <= row.swing_end_id)
    ]
    start_datetime = df.iloc[0]["start_datetime"]
    end_datetime = df.iloc[-1]["end_datetime"]
    df_trend.at[i, "start_datetime"] = start_datetime
    df_trend.at[i, "end_datetime"] = end_datetime

xs_trend = []
ys_trend = []
texts_trend = []
text_positions_trend = []
for i, row in enumerate(df_trend.itertuples()):
    xs_trend += [row.start_datetime, row.end_datetime, None]
    print("df_trend[marker] - "+ str(i), row)
    start_index = df_swing[(df_swing["id"] == row.swing_start_id)]["index"].iloc[0]
    end_index = df_swing[(df_swing["id"] == row.swing_end_id)]["index"].iloc[0]
    if row.direction == Direction.UP:
        start_price = df_swing[(df_swing["id"] == row.swing_start_id)]["low_price"].iloc[0]
        end_price = df_swing[(df_swing["id"] == row.swing_end_id)]["high_price"].iloc[0]
        text_positions_trend += ["bottom center", "top center", "middle center"]
    else:
        start_price = df_swing[(df_swing["id"] == row.swing_start_id)]["high_price"].iloc[
            0
        ]
        end_price = df_swing[(df_swing["id"] == row.swing_end_id)]["low_price"].iloc[0]
        text_positions_trend += ["top center", "bottom center", "middle center"]
    ys_trend += [start_price, end_price, None]
    texts_trend += [start_index, end_index, None]
#endregion

# region fig
fig = make_subplots(rows=1, cols=1)
fig.add_trace(
    go.Candlestick(
        x=df_sbar.index,
        open=df_sbar["open_price"],
        high=df_sbar["high_price"],
        low=df_sbar["low_price"],
        close=df_sbar["close_price"],
        name="K线",
    ),
    row=1,
    col=1,
)
#
# df_fractal_bottom = df_sbar[(df_sbar["fractal_type"] == FractalType.BOTTOM)]
#
# fig.add_trace(
#     go.Scatter(
#         x=df_fractal_bottom["datetime"],
#         y=df_fractal_bottom["low_price"] * 0.995,  # 放在K线最低价下方一点，不遮挡蜡烛
#         mode="markers+text",
#         name="swing point low",
#         marker=dict(
#             symbol="triangle-up",
#             size=14,
#             color="#1E90FF",
#         ),
#         text=df_fractal_bottom["index"],  # ← 添加序号
#         textposition="bottom center",  # ← 文字位置
#         textfont=dict(size=10, color="white"),
#         hovertemplate="<b>波段低点</b><br>"
#         + "时间: %{x}<br>"
#         + "价格: %{y:.2f}<br>"
#         + "<extra></extra>",
#     ),
#     row=1,
#     col=1,
# )
#
#
# df_fractal_top = df_sbar[(df_sbar["fractal_type"] == FractalType.TOP)]
#
# fig.add_trace(
#     go.Scatter(
#         x=df_fractal_top["datetime"],
#         y=df_fractal_top["high_price"] * 1.005,  # 放在K线最高价上方一点
#         mode="markers+text",
#         name="swing point high",
#         marker=dict(
#             symbol="triangle-down",
#             size=14,
#             color="#FF4500",
#         ),
#         text=df_fractal_top["index"],  # ← 添加序号
#         textposition="top center",  # ← 文字位置
#         textfont=dict(size=10, color="white"),
#         hovertemplate="<b>波段高点</b><br>"
#         + "时间: %{x}<br>"
#         + "价格: %{y:.2f}<br>"
#         + "<extra></extra>",
#     ),
#     row=1,
#     col=1,
# )

fig.add_trace(
    go.Scatter(
        x=xs_swing,
        y=ys_swing,
        name="swing segments",
        mode="lines",  # lines+text 支持显示文字
        line=dict(width=2, color="orange"),
        text=texts_swing,
        textposition=text_positions_swing,  # 文字位置
    )
)

fig.add_trace(
    go.Scatter(
        x=xs_trend,
        y=ys_trend,
        name="trend segments",
        mode="lines",  # lines+text 支持显示文字
        line=dict(width=2, color="red"),
        text=texts_trend,
        textposition=text_positions_trend,  # 文字位置
    )
)

keyzone_manager = KeyZoneManager(mtc)
df_keyzone = keyzone_manager.read_parquet()
df_keyzone = df_keyzone.filter(pl.col("timeframe")==timeframe)
zones = [KeyZone(**row) for row in df_keyzone.rows(named=True)]
for zone in zones:
    print(zone)
    sbar_list = sbar_manager.get_around_sbar(zone.sbar_start_id, 10)
    start_dt = sbar_list[0].datetime
    end_dt = sbar_list[-1].datetime
    fig.add_shape(
        type="rect",
        xref="x",
        yref="y",
        x0=start_dt,
        x1=end_dt,
        y0=zone.lower,
        y1=zone.upper,
        fillcolor="rgba(0, 100, 255, 0.2)",
        line=dict(width=0)
    )

fig.update_layout(
    title="KeyZone Chart",
    height=900,
    hovermode="x unified",  # X轴悬停联动虚线
)

fig.update_xaxes(
    showgrid=False,
)

fig.update_yaxes(
    showgrid=False,
)
def build_range_breaks(start_dt, end_dt):

    # 添加节假日
    holidays = [
        d.strftime("%Y-%m-%d")
        # pd.to_datetime(d)
        for d in cc.get_holidays(start_dt, end_dt, include_weekends=False)
    ]
    print(holidays)
    if holidays:
        holidays = dict(values=holidays, dvalue=24*60*60*1000)

    range_breaks = [
        dict(bounds=["sat", "mon"]),
        # # 午休
        dict(bounds=[11.51, 13.5], pattern="hour"),
        #
        # # 小节间休息
        dict(bounds=[10.26, 10.5], pattern="hour"), # 10+15/60 = 10.25
        #
        # # 非交易时段（适用于有夜盘品种）
        dict(bounds=[15.01, 21], pattern="hour"),    # 白盘收盘到夜盘开盘
        dict(bounds=[23.01, 9], pattern="hour"),    # 夜盘23点后到次日9点

        # 节假日
        # dict(values=['2024-02-10', '2024-02-11', '2024-02-12', '2024-02-13', '2024-02-14'])
        # holidays
    ]
    print(range_breaks)

    return range_breaks

fig.update_xaxes(
        rangebreaks= build_range_breaks(df_sbar.index[0],df_sbar.index[-1]),
)

fig.show()
# endregion

# endregion

In [ ]:
from typing import List
import akshare as ak
import polars as pl
from pulao.bar import SBar

def get_around_sbar(
    self, pivot_id: int, length: int, ret_df: bool = False
) -> List[SBar] | None | pl.DataFrame:
    """
    获取在pivot_id周围的sbar list
    :param pivot_id: 中轴点
    :param length: 左右各多少根
    :param ret_df: 是否返回原生pl.DataFrame
    :return:
    """
    idx = self.get_index(pivot_id)
    if idx is None:
        assert 1==2 , f"id：{pivot_id} 不在df_sbar中，不应该出现这个情况"

    start_index = idx - length
    end_index = idx + length

    if start_index < 0:
        start_index = 0
    if end_index >= self.df_sbar.height:
        end_index = self.df_sbar.height - 1

    df = self.df_sbar.slice(start_index, end_index - start_index + 1)
    if ret_df:
        return df

    if df.is_empty():
        return None
    return [SBar(**row) for row in df.rows(named=True)]

# get_around_sbar(sbar_manager, 123789819393343488,10)
sbar_manager.df_sbar.filter(pl.col("id") < 123789819393343488)

In [ ]:
trend_manager.pretty_worker_id()

In [ ]:
# 可视化日志
import pandas as pd

df = pd.read_json("./logs/pulao.log", lines=True)
df = df.drop(columns=["logger", "level", "time"])
priority_cols = ["event"]
other_cols = [c for c in df.columns if c not in priority_cols]
df = df[priority_cols + other_cols]
df["info"] = df[other_cols].apply(lambda row: row.dropna().tolist(), axis=1)
df = df.drop(columns=other_cols)
df["info"] = df["info"].apply(lambda x: "" if x == [] else x)
df